# gigapath_runpod (로컬→RunPod SSH 오케스트레이션)

로컬 주피터 노트북에서 SSH/rsync로 RunPod 인스턴스를 제어하며 **SVS 업로드 → 타일링 → 원본 삭제 → 벡터화 → 타일 삭제 → 학습** 순서로 스토리지 사용을 최소화합니다. 아래 자리표시자(타일링/벡터화/학습 커맨드, 호스트명 등)를 환경에 맞게 수정하세요.


## 0. 기본 설정 (호스트/경로/커맨드 자리표시자)

- `SSH_HOST_DIRECT/PORT`: RunPod Direct TCP (SCP/rsync 가능)
- `SSH_HOST_GATEWAY`: 필요 시 게이트웨이 사용 (SCP 제한)
- `REMOTE_RAW`, `REMOTE_WORK`: RunPod 내부 경로
- 타일링/벡터화/학습 커맨드를 gigapath 코드에 맞게 채우기
- 삭제 정책 토글: `DELETE_SVS_AFTER_TILING`, `DELETE_TILES_AFTER_FEATURE`


In [1]:

import subprocess, shlex, time
from pathlib import Path

# SSH 대상 및 키 (실제 RunPod 접속 정보로 설정)
# 게이트웨이(ssh.runpod.io)와 Direct TCP(root@IP -p PORT)를 모두 지원
SSH_HOST_GATEWAY = "crxrrtk7qj79so-64411c13@ssh.runpod.io"  # proxied, SCP 제한
SSH_HOST_DIRECT = "root@216.81.151.15"  # direct TCP, SCP/rsync 가능
SSH_PORT_DIRECT = 13458
SSH_KEY = "~/.ssh/runpod_peter"  # 필요 없으면 None
# 기본은 Direct TCP를 사용(PTY 오류 회피, rsync/ssh 모두 동일 경로)
USE_DIRECT_FOR_SSH = True
USE_DIRECT_FOR_RSYNC = True
# PTY 옵션: direct에서는 비워두고, 게이트웨이 필요 시 설정 (예: "-T")
SSH_EXTRA_OPTS = ""

# RunPod 내부 경로
REMOTE_RAW = "~/data/raw"
REMOTE_WORK = "~/data/work"
REMOTE_CHECKPOINT = f"{REMOTE_WORK}/checkpoints"
REMOTE_LOG = "~/logs"

# 타일링/벡터화/학습 커맨드 템플릿 (도구 파일 없이 인라인 tiler 사용)
TILE_SIZE = 224
TILE_OVERLAP = 0
TILING_CMD_TEMPLATE = r"""
python - <<'PY'
import openslide
from pathlib import Path
from PIL import Image
import numpy as np
from skimage import color, filters

PATCH_SIZE = {tile_size}
MIN_TISSUE_RATIO = 0.5
LEVEL = 0

svs_path = Path("{input}").expanduser()
out_dir = Path("{output}").expanduser()
out_dir.mkdir(parents=True, exist_ok=True)

slide = openslide.OpenSlide(str(svs_path))
width, height = slide.level_dimensions[LEVEL]
est = ((height + PATCH_SIZE - 1) // PATCH_SIZE) * ((width + PATCH_SIZE - 1) // PATCH_SIZE)
print("tiling {{}}: expected {{}} tiles".format(svs_path.name, est))

def tissue_mask(rgb):
    hsv = color.rgb2hsv(rgb)
    sat = hsv[:, :, 1]
    thresh = filters.threshold_otsu(sat)
    return sat > thresh

count = 0
for y in range(0, height, PATCH_SIZE):
    for x in range(0, width, PATCH_SIZE):
        region = slide.read_region((x, y), LEVEL, (PATCH_SIZE, PATCH_SIZE)).convert("RGB")
        arr = np.array(region)
        if tissue_mask(arr).mean() < MIN_TISSUE_RATIO:
            continue
        tile_name = "tile_{{:06d}}_x{{}}_y{{}}.png".format(count, x, y)
        region.save(out_dir / tile_name, format="PNG")
        count += 1
slide.close()
print("done: {{}} tiles saved to {{}}".format(count, out_dir))
PY
"""
FEATURE_CMD_TEMPLATE = "python tools/extract_features.py --tiles {tiles} --out {out}"
TRAIN_CMD_TEMPLATE = (
    "python train.py --features {features_dir} --out {ckpt_dir} --logdir {log_dir}"
)

# 삭제 정책
DELETE_SVS_AFTER_TILING = True
DELETE_TILES_AFTER_FEATURE = True

print("SSH_HOST_DIRECT=", SSH_HOST_DIRECT, "port", SSH_PORT_DIRECT)
print("SSH_HOST_GATEWAY=", SSH_HOST_GATEWAY)
print("REMOTE_RAW=", REMOTE_RAW)
print("REMOTE_WORK=", REMOTE_WORK)
print("TILING_CMD_TEMPLATE=", TILING_CMD_TEMPLATE)
print("FEATURE_CMD_TEMPLATE=", FEATURE_CMD_TEMPLATE)
print("TRAIN_CMD_TEMPLATE=", TRAIN_CMD_TEMPLATE)


SSH_HOST_DIRECT= root@216.81.151.15 port 13458
SSH_HOST_GATEWAY= crxrrtk7qj79so-64411c13@ssh.runpod.io
REMOTE_RAW= ~/data/raw
REMOTE_WORK= ~/data/work
TILING_CMD_TEMPLATE= 
python - <<'PY'
import openslide
from pathlib import Path
from PIL import Image
import numpy as np
from skimage import color, filters

PATCH_SIZE = {tile_size}
MIN_TISSUE_RATIO = 0.5
LEVEL = 0

svs_path = Path("{input}").expanduser()
out_dir = Path("{output}").expanduser()
out_dir.mkdir(parents=True, exist_ok=True)

slide = openslide.OpenSlide(str(svs_path))
width, height = slide.level_dimensions[LEVEL]
est = ((height + PATCH_SIZE - 1) // PATCH_SIZE) * ((width + PATCH_SIZE - 1) // PATCH_SIZE)
print("tiling {{}}: expected {{}} tiles".format(svs_path.name, est))

def tissue_mask(rgb):
    hsv = color.rgb2hsv(rgb)
    sat = hsv[:, :, 1]
    thresh = filters.threshold_otsu(sat)
    return sat > thresh

count = 0
for y in range(0, height, PATCH_SIZE):
    for x in range(0, width, PATCH_SIZE):
        region = slide.read_

## 1. 유틸리티: 로컬 실행/SSH/rsync 래퍼

- `run_local`: 로컬 셸 실행
- `run_ssh`: RunPod에서 명령 실행
- `rsync_upload` / `rsync_download`: 부분 업로드/다운로드 지원


In [2]:
def run_local(cmd, check=True):
    print(f"[local] $ {cmd}")
    # capture stdout/stderr so rsync/ssh 에러 메시지를 바로 확인
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if check and result.returncode != 0:
        msg = result.stderr.strip() or result.stdout.strip()
        raise RuntimeError(f"Local command failed ({result.returncode}): {cmd}{msg}")
    return result.returncode

def _ssh_parts(host, port=None):
    parts = ["ssh"]
    if SSH_KEY:
        parts += ["-i", SSH_KEY]
    if SSH_EXTRA_OPTS:
        parts += shlex.split(SSH_EXTRA_OPTS)
    if port:
        parts += ["-p", str(port)]
    parts += [host]
    return parts

def run_ssh(cmd, check=True, use_direct=None):
    if use_direct is None:
        use_direct = USE_DIRECT_FOR_SSH
    parts = _ssh_parts(
        SSH_HOST_DIRECT if use_direct else SSH_HOST_GATEWAY,
        SSH_PORT_DIRECT if use_direct else None,
    )
    ssh_cmd = " ".join(shlex.quote(p) for p in parts + [cmd])
    print(f"[ssh] $ {cmd}")
    result = subprocess.run(ssh_cmd, shell=True, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if check and result.returncode != 0:
        msg = result.stderr.strip() or result.stdout.strip()
        raise RuntimeError(f"SSH command failed ({result.returncode}): {cmd}{msg}")
    return result.returncode



def _rsync_ssh_opt(use_direct=True):
    # Build ssh transport for rsync (-e). Host is kept separate to avoid being injected into -e.
    key = Path(SSH_KEY).expanduser() if SSH_KEY else None
    parts = ["ssh"]
    if key:
        parts += ["-i", str(key)]
    if SSH_EXTRA_OPTS:
        parts += shlex.split(SSH_EXTRA_OPTS)
    if use_direct:
        parts += ["-p", str(SSH_PORT_DIRECT)]
    return " ".join(parts)


def rsync_upload(local_path: Path, remote_dir: str):
    use_direct = USE_DIRECT_FOR_RSYNC
    ssh_opt = _rsync_ssh_opt(use_direct)
    remote_host = SSH_HOST_DIRECT if use_direct else SSH_HOST_GATEWAY
    remote_target = f"{remote_host}:{remote_dir.rstrip('/')}/"
    cmd = (
        # f"rsync -avh --partial --info=progress2 -e {shlex.quote(ssh_opt)} "
        f"rsync -avh --partial --progress -e {shlex.quote(ssh_opt)} "
        f"{shlex.quote(str(local_path))} {remote_target}"
    )
    run_local(cmd)


def rsync_download(remote_path: str, local_dir: Path):
    use_direct = USE_DIRECT_FOR_RSYNC
    ssh_opt = _rsync_ssh_opt(use_direct)
    remote_host = SSH_HOST_DIRECT if use_direct else SSH_HOST_GATEWAY
    local_dir.mkdir(parents=True, exist_ok=True)
    remote_src = f"{remote_host}:{remote_path}"
    cmd = (
        f"rsync -avh --partial --info=progress2 -e {shlex.quote(ssh_opt)} "
        f"rsync -avh --partial --progress -e {shlex.quote(ssh_opt)} "
        f"{remote_src} {shlex.quote(str(local_dir))}/"
    )
    run_local(cmd)


## 2. 원격 기본 디렉토리 준비 및 점검

- 필요한 폴더 생성
- GPU/디스크 확인 (실패 시에도 진행되도록 `|| true`)
- `ssh` 접속 테스트까지 포함


In [3]:
# 접속 및 경로 생성
run_ssh(f"mkdir -p {REMOTE_RAW} {REMOTE_WORK} {REMOTE_CHECKPOINT} {REMOTE_LOG}")
run_ssh("echo 'SSH OK on $(hostname)' && nvidia-smi || true", check=False)
run_ssh("df -h . || true", check=False)


[ssh] $ mkdir -p ~/data/raw ~/data/work ~/data/work/checkpoints ~/logs
[ssh] $ echo 'SSH OK on $(hostname)' && nvidia-smi || true
SSH OK on $(hostname)
Thu Nov 27 05:25:47 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.133.20             Driver Version: 570.133.20     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX A6000               On  |   00000000:0E:00.0 Off |                  Off |
| 30%   29C    P8             27W /  300W |       0MiB /  49140MiB |      0%    

0

## 2.5 원격 의존성 설치 (tiling용 openslide 등)

- RunPod에 openslide/python 패키지가 없을 때 실행
- 최소 의존성만 설치: openslide-python, pillow, numpy, scikit-image


In [4]:
REMOTE_PY_PKGS = ['openslide-python', 'Pillow', 'numpy', 'scikit-image']
REMOTE_VENV = '~/venv_gigapath'

RUN_REMOTE_SETUP = True

def install_remote_deps(pkgs=None, venv_path=REMOTE_VENV):
    pkgs = pkgs or REMOTE_PY_PKGS
    pkg_str = ' '.join(pkgs)
    cmd = (
        f'python3 -m venv {venv_path} && ' +
        f'{venv_path}/bin/pip install --upgrade pip && ' +
        f'{venv_path}/bin/pip install {pkg_str}'
    )
    print('[remote setup] installing into venv:', venv_path, '| pkgs:', pkg_str)
    run_ssh(cmd)

if RUN_REMOTE_SETUP:
    install_remote_deps()
else:
    print('Remote dependency install skipped (set RUN_REMOTE_SETUP=True)')


[remote setup] installing into venv: ~/venv_gigapath | pkgs: openslide-python Pillow numpy scikit-image
[ssh] $ python3 -m venv ~/venv_gigapath && ~/venv_gigapath/bin/pip install --upgrade pip && ~/venv_gigapath/bin/pip install openslide-python Pillow numpy scikit-image



## 3. 데이터 선택: mammary adenoma/adenocarcinoma 100개씩 (외장 2023)

- `mammary_adenoma_vs_adenocarcinoma_only(2023).parquet`에서 라벨을 읽어 두 클래스만 필터링
- `/Volumes/Expansion/2023/<FOLDER>/<FILE_NAME>.svs` 존재 여부 확인 후 클래스별 100개씩 셔플 샘플링
- 선택 결과는 `selected_slides` 리스트에 `slide_id/path/label` 필드로 저장


In [5]:
import random
import pandas as pd

SOURCE_SLIDE_ROOT = Path("/Volumes/Expansion/2023")
LABEL_PARQUET = Path("/Users/curv/Repos/GC-Pathology/PoC/v1/mammary_adenoma_vs_adenocarcinoma_only(2023).parquet")
SAMPLES_PER_CLASS = 100
LABEL_MAP = {"mammary_adenoma": 0, "mammary_adenocarcinoma": 1}

def _normalize_labels(df: pd.DataFrame) -> pd.DataFrame:
    label_norm = df["label"].astype(str).str.lower().str.strip()
    filtered = df.loc[label_norm.isin(LABEL_MAP.keys())].copy()
    filtered["label"] = label_norm.loc[filtered.index].map(LABEL_MAP)
    return filtered

def _collect(rows: pd.DataFrame, per_class: int):
    rows = rows.sample(frac=1, random_state=42).to_dict("records")
    picked = []
    for row in rows:
        slide_dir = SOURCE_SLIDE_ROOT / str(row["FOLDER"]).strip()
        file_names = [fn.strip() for fn in str(row["FILE_NAME"]).split("|") if fn.strip()]
        slide_id = row.get("INSP_RQST_NO", row.get("INSP_RQST_NUM", row["FOLDER"]))
        for file_name in file_names:
            fn = file_name if file_name.lower().endswith(".svs") else f"{file_name}.svs"
            slide_path = slide_dir / fn
            if slide_path.exists():
                picked.append({"slide_id": slide_id, "path": slide_path, "label": int(row["label"])})
                break
        if len(picked) >= per_class:
            break
    return picked

df_labels = _normalize_labels(pd.read_parquet(LABEL_PARQUET))
pos_rows = df_labels[df_labels["label"] == 1]
neg_rows = df_labels[df_labels["label"] == 0]
if pos_rows.empty or neg_rows.empty:
    raise ValueError(f"라벨 분포 확인 필요: pos={len(pos_rows)}, neg={len(neg_rows)}")

pos_pick = _collect(pos_rows, SAMPLES_PER_CLASS)
neg_pick = _collect(neg_rows, SAMPLES_PER_CLASS)
if len(pos_pick) < SAMPLES_PER_CLASS or len(neg_pick) < SAMPLES_PER_CLASS:
    print(f"경고: 요청 개수보다 부족 (pos {len(pos_pick)}/{SAMPLES_PER_CLASS}, neg {len(neg_pick)}/{SAMPLES_PER_CLASS})")

selected_slides = pos_pick[:SAMPLES_PER_CLASS] + neg_pick[:SAMPLES_PER_CLASS]
random.shuffle(selected_slides)

pos_cnt = sum(rec["label"] for rec in selected_slides)
neg_cnt = len(selected_slides) - pos_cnt
print(f"선정된 슬라이드: {len(selected_slides)}개 (pos={pos_cnt}, neg={neg_cnt})")
print("예시 3개:", selected_slides[:3])


선정된 슬라이드: 200개 (pos=100, neg=100)
예시 3개: [{'slide_id': '20230815-108-0007', 'path': PosixPath('/Volumes/Expansion/2023/S23-04904/S23-04904#1#A##9.svs'), 'label': 1}, {'slide_id': '20231108-150-0001', 'path': PosixPath('/Volumes/Expansion/2023/S23-07403/S23-07403#1###8.svs'), 'label': 0}, {'slide_id': '20231027-128-0005', 'path': PosixPath('/Volumes/Expansion/2023/S23-06947/S23-06947#1#A##3.svs'), 'label': 1}]


## 4. 원격 타일링 함수 (SVS 1개)

- RunPod에서 타일링 후 옵션에 따라 원본 삭제
- `TILING_CMD_TEMPLATE`를 gigapath에 맞게 수정


In [6]:
def remote_tile(svs_remote_path: str) -> str:
    svs_name = Path(svs_remote_path).name
    tile_dir = f"{REMOTE_WORK}/{Path(svs_name).stem}_tiles"
    cmd = TILING_CMD_TEMPLATE.format(
        input=svs_remote_path,
        output=tile_dir,
        tile_size=TILE_SIZE,
        overlap=TILE_OVERLAP,
    )
    # run tiler with venv python if available
    if "python - <<'PY'" in cmd and 'REMOTE_VENV' in globals():
        cmd = cmd.replace("python - <<'PY'", f"{REMOTE_VENV}/bin/python - <<'PY'")
    run_ssh(cmd)
    if DELETE_SVS_AFTER_TILING:
        run_ssh(f"rm -f {svs_remote_path}")
        print(f"Deleted original svs: {svs_remote_path}")
    return tile_dir


## 5. 타일 벡터화 템플릿 (GigaPath)

- timm `prov-gigapath/prov-gigapath`로 타일 임베딩 생성
- FEATURE_CMD_TEMPLATE를 정의하고 remote_featurize에서 사용
- 저장 후 최종 npy 파일 크기를 MB 단위로 출력


In [7]:
FEATURE_CMD_TEMPLATE = r"""
python - <<'PY'
import torch
import timm
from timm.data import resolve_model_data_config, create_transform
from pathlib import Path
from PIL import Image
import numpy as np
from tqdm import tqdm

tiles_dir = Path("{tiles}").expanduser()
out_path = Path("{out}").expanduser()
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)
if device != 'cuda':
    raise SystemExit('CUDA device not available for feature extraction')
torch.backends.cudnn.benchmark = True

model = timm.create_model('hf_hub:prov-gigapath/prov-gigapath', pretrained=True).to(device)
model.eval()
config = resolve_model_data_config(model)
transform = create_transform(**config, is_training=False)

tile_paths = sorted(tiles_dir.glob('*.png'))
if not tile_paths:
    raise SystemExit(f'No tiles found in {tiles_dir}')

batch_size = 32
feats = []
with torch.inference_mode():
    for i in range(0, len(tile_paths), batch_size):
        batch_files = tile_paths[i:i+batch_size]
        imgs = [transform(Image.open(p).convert('RGB')) for p in batch_files]
        batch = torch.stack(imgs).to(device)
        emb = model(batch)
        if isinstance(emb, (tuple, list)):
            emb = emb[0]
        feats.append(emb.detach().cpu().numpy())
feat_arr = np.concatenate(feats, axis=0)
np.save(out_path, feat_arr)
size_mb = out_path.stat().st_size / (1024*1024)
print(f'saved {feat_arr.shape} -> {out_path} ({size_mb:.1f} MB)')
PY
"""

def remote_featurize(tile_dir: str) -> str:
    feat_path = f'{tile_dir}_feats.npy'
    cmd = FEATURE_CMD_TEMPLATE.format(tiles=tile_dir, out=feat_path)
    if "python - <<'PY'" in cmd and 'REMOTE_VENV' in globals():
        cmd = cmd.replace("python - <<'PY'", REMOTE_VENV + "/bin/python - <<'PY'")
    run_ssh(cmd)
    if DELETE_TILES_AFTER_FEATURE:
        run_ssh(f'rm -rf {tile_dir}')
        print(f'Deleted tiles: {tile_dir}')
    print(f'Feature saved: {feat_path}')
    return feat_path


## 6. 선택 슬라이드 업로드→타일→벡터화 (순차 처리)

- `selected_slides`(각 클래스 최대 100개)를 순서대로 업로드/처리
- 슬라이드 1개 단위로 rsync 업로드 → 타일링 → 벡터화 → 옵션에 따라 원본/타일 삭제
- `RUN_SLIDE_PROCESSING` 토글로 실행 여부를 제어하고, `PROCESS_LIMIT`로 일부만 시범 실행 가능


In [8]:
def process_selected_slides(slides=None, limit=None):
    slides = slides or selected_slides
    if not slides:
        print("selected_slides가 비어있습니다. 3단계 데이터 선택 셀을 실행하세요.")
        return []

    to_run = slides if limit is None else slides[:limit]
    results = []
    for idx, rec in enumerate(to_run, 1):
        local_path = Path(rec["path"])
        remote_svs = f"{REMOTE_RAW}/{local_path.name}"
        print(f"\n[{idx}/{len(to_run)}] {local_path.name} (label={rec['label']})")
        rsync_upload(local_path, REMOTE_RAW)
        start = time.time()
        tile_dir = remote_tile(remote_svs)
        feat_path = remote_featurize(tile_dir)
        elapsed = time.time() - start
        results.append({
            **rec,
            "remote_svs": remote_svs,
            "feature_path": feat_path,
            "elapsed_min": elapsed / 60,
        })
        print(f"완료: {local_path.name} -> {feat_path} ({elapsed/60:.1f} min)")
    return results

RUN_SLIDE_PROCESSING = True
PROCESS_LIMIT = None  # 예: 10 으로 설정하면 앞 10개만 처리

if RUN_SLIDE_PROCESSING:
    processed_features = process_selected_slides(limit=PROCESS_LIMIT)
else:
    print("Processing skipped (RUN_SLIDE_PROCESSING=True 로 순차 실행)")



[1/200] S23-04904#1#A##9.svs (label=1)
[local] $ rsync -avh --partial --progress -e 'ssh -i /Users/curv/.ssh/runpod_peter -p 13458' '/Volumes/Expansion/2023/S23-04904/S23-04904#1#A##9.svs' root@216.81.151.15:~/data/raw/
Transfer starting: 1 files
S23-04904#1#A##9.svs
          32768   0%    1.21KB/s   243:26:30
        2162688   0%    1.74MB/s   00:09:53
        2392064   0%  446.39KB/s   00:39:25
        3014656   0%  972.98KB/s   00:18:04
        5734400   1%    5.19MB/s   00:03:18
        9732096   1%    7.62MB/s   00:02:14
       13828096   1%    7.81MB/s   00:02:10
       17825792   2%    7.60MB/s   00:02:13
       21889024   2%    7.65MB/s   00:02:12
       26050560   2%    7.86MB/s   00:02:08
       30212096   3%    7.81MB/s   00:02:08
       34308096   3%    7.79MB/s   00:02:08
       37781504   3%    6.44MB/s   00:02:34
       41910272   4%    7.81MB/s   00:02:07
       46104576   4%    7.50MB/s   00:02:12
       50298880   5%    7.76MB/s   00:02:06
       54362112   5%    7.

KeyboardInterrupt: 

## 7. 원격 학습 실행 (옵션)

- `RUN_TRAINING=True`로 토글
- `TRAIN_CMD_TEMPLATE`를 gigapath 학습 스크립트에 맞게 수정
- 체크포인트/로그 경로는 REMOTE_CHECKPOINT/REMOTE_LOG


In [ ]:
RUN_TRAINING = False  # True로 바꾸면 학습 실행

def run_remote_training():
    train_cmd = TRAIN_CMD_TEMPLATE.format(
        features_dir=shlex.quote(REMOTE_WORK),
        ckpt_dir=shlex.quote(REMOTE_CHECKPOINT),
        log_dir=shlex.quote(REMOTE_LOG),
    )
    print(f"Training command:{train_cmd}")
    run_ssh(train_cmd)

if RUN_TRAINING:
    run_remote_training()
else:
    print("Training skipped (set RUN_TRAINING=True to run)")


## 8. 결과 다운로드/아카이브

- 필요 체크포인트/feature를 로컬로 rsync 다운로드
- 오래된 체크포인트는 RunPod에서 압축 후 삭제


In [10]:
def download_checkpoints(local_dir: Path):
    rsync_download(f"{REMOTE_CHECKPOINT}/", local_dir)

def remote_archive_and_delete(path_glob: str):
    # 예: path_glob="~/data/work/checkpoints/epoch*"
    cmd = f"for p in {path_glob}; do [ -e "$p" ] || continue; tar -czf ${p}.tar.gz -C $(dirname $p) $(basename $p) && rm -rf $p; done"
    run_ssh(cmd)


SyntaxError: invalid syntax (1354155642.py, line 6)

## 9. 추천 워크플로우 (셀 실행 순서)

1) 0~2 셀 실행: 설정/접속/경로 준비
2) 2.5 셀로 원격 의존성 설치 필요 시 실행(RUN_REMOTE_SETUP=True)
3) 3 셀 실행: 외장 `/Volumes/Expansion/2023`에서 클래스별 100개 슬라이드 샘플링(`selected_slides`)
4) 4~5 셀 실행: 타일링/벡터화 함수 정의(FEATURE_CMD_TEMPLATE 포함)
5) 6 셀에서 `RUN_SLIDE_PROCESSING=True` 설정 후 실행: 업로드→타일→벡터 순차 처리
6) 필요 시 7 셀에서 학습 실행, 8 셀로 결과 다운로드/정리

공간 절약: 타일링 후 원본 삭제, 벡터화 후 타일 삭제, 체크포인트는 주기적으로 압축/정리.
